# 🤖 Classification — E1–E7

Dự đoán `is_modified` (code AI có bị sửa hay không) từ kích thước code, repo, ngôn ngữ, công cụ AI, task và thời gian. Đây là bài toán dự đoán, không phải chứng minh nguyên nhân.

Cần cài thêm: `scikit-learn`.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, RocCurveDisplay
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
DATA_PATH = Path('../data/processed/commits_features.csv')
REPORT_DIR = Path('../reports'); FIG_DIR = REPORT_DIR / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH, low_memory=False).replace([np.inf,-np.inf],np.nan)
feature_cols = ['code_lines','code_chars','repo_commit_count','language','ai_tool','ai_verb','task_type','year','hour','is_weekend']
df = df.dropna(subset=feature_cols + ['is_modified']).copy()
X = df[feature_cols]; y = df['is_modified'].astype(int)
numeric_features = ['code_lines','code_chars','repo_commit_count','year','hour']
categorical_features = ['language','ai_tool','ai_verb','task_type','is_weekend']
preprocessor = ColumnTransformer([('num', StandardScaler(), numeric_features), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])
models = {'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced'), 'Decision Tree': DecisionTreeClassifier(max_depth=12, min_samples_leaf=10, class_weight='balanced', random_state=42), 'Random Forest': RandomForestClassifier(n_estimators=250, max_depth=16, min_samples_leaf=3, class_weight='balanced', n_jobs=-1, random_state=42)}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, stratify=y, random_state=42)
print(f'Dataset: {len(df):,} dòng; train={len(X_train):,}; test={len(X_test):,}; positive rate={y.mean():.3f}')

Dataset: 33,917 dòng; train=27,133; test=6,784; positive rate=0.371


## E1–E4 — Train/test và huấn luyện 3 mô hình

In [2]:
pipelines = {name: Pipeline([('preprocess', preprocessor), ('model', model)]) for name, model in models.items()}
fitted = {}
predictions = {}
probabilities = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train); fitted[name] = pipe
    predictions[name] = pipe.predict(X_test); probabilities[name] = pipe.predict_proba(X_test)[:,1]
    print(f'--- {name} ---'); print(classification_report(y_test, predictions[name], target_names=['Not modified','Modified']))

--- Logistic Regression ---
              precision    recall  f1-score   support

Not modified       0.69      0.70      0.69      4266
    Modified       0.48      0.47      0.48      2518

    accuracy                           0.61      6784
   macro avg       0.59      0.58      0.59      6784
weighted avg       0.61      0.61      0.61      6784

--- Decision Tree ---
              precision    recall  f1-score   support

Not modified       0.75      0.72      0.74      4266
    Modified       0.56      0.60      0.58      2518

    accuracy                           0.68      6784
   macro avg       0.66      0.66      0.66      6784
weighted avg       0.68      0.68      0.68      6784

--- Random Forest ---
              precision    recall  f1-score   support

Not modified       0.82      0.84      0.83      4266
    Modified       0.72      0.70      0.71      2518

    accuracy                           0.79      6784
   macro avg       0.77      0.77      0.77      6784
we

## E5 — Accuracy, Precision, Recall, F1, ROC-AUC và confusion matrix

In [3]:
metric_rows = []
for name in models:
    pred, prob = predictions[name], probabilities[name]
    metric_rows.append({'model':name, 'accuracy':accuracy_score(y_test,pred), 'precision':precision_score(y_test,pred), 'recall':recall_score(y_test,pred), 'f1':f1_score(y_test,pred), 'roc_auc':roc_auc_score(y_test,prob)})
metrics = pd.DataFrame(metric_rows).sort_values('roc_auc', ascending=False)
display(metrics); metrics.to_csv(REPORT_DIR / 'classification_metrics.csv', index=False)
fig, axes = plt.subplots(1,3,figsize=(15,4))
for ax,(name,pred) in zip(axes,predictions.items()): ConfusionMatrixDisplay(confusion_matrix(y_test,pred), display_labels=['Not modified','Modified']).plot(ax=ax, colorbar=False); ax.set_title(name)
fig.tight_layout(); fig.savefig(FIG_DIR / 'E5_confusion_matrices.png', dpi=160, bbox_inches='tight'); plt.show(); plt.close(fig)
fig, ax = plt.subplots(figsize=(7,5))
for name,prob in probabilities.items(): RocCurveDisplay.from_predictions(y_test,prob,name=name,ax=ax)
ax.set_title('ROC curves'); fig.tight_layout(); fig.savefig(FIG_DIR / 'E5_roc_curves.png', dpi=160, bbox_inches='tight'); plt.show(); plt.close(fig)

,model,accuracy,precision,recall,f1,roc_auc
2,Random Forest,0.786557,0.719442,0.696585,0.707829,0.857655
1,Decision Tree,0.678213,0.562245,0.600874,0.580918,0.735708
0,Logistic Regression,0.613355,0.478958,0.474583,0.476760,0.619145


## E6 — Feature importance

In [4]:
importance_rows = []
for name, pipe in fitted.items():
    model = pipe.named_steps['model']; prep = pipe.named_steps['preprocess']; names = prep.get_feature_names_out()
    values = model.coef_[0] if hasattr(model,'coef_') else model.feature_importances_
    for feature, value in zip(names, values): importance_rows.append({'model':name, 'feature':feature, 'importance':value, 'abs_importance':abs(value)})
importance = pd.DataFrame(importance_rows)
display(importance.sort_values(['model','abs_importance'], ascending=[True,False]).groupby('model').head(15))
importance.to_csv(REPORT_DIR / 'classification_feature_importance.csv', index=False)
rf_top = importance[importance.model.eq('Random Forest')].nlargest(15,'abs_importance').sort_values('importance')
plt.figure(figsize=(9,6)); sns.barplot(data=rf_top,x='importance',y='feature',color='#16a34a'); plt.title('Top 15 Random Forest feature importance'); plt.xlabel('Importance'); plt.ylabel('Feature'); plt.tight_layout(); plt.savefig(FIG_DIR / 'E6_feature_importance.png',dpi=160,bbox_inches='tight'); plt.show(); plt.close()

,model,feature,importance,abs_importance
31,Decision Tree,num__code_chars,0.252023,0.252023
32,Decision Tree,num__repo_commit_count,0.224377,0.224377
30,Decision Tree,num__code_lines,0.167795,0.167795
34,Decision Tree,num__hour,0.130524,0.130524
33,Decision Tree,num__year,0.052017,0.052017
50,Decision Tree,cat__ai_verb_written,0.016241,0.016241
37,Decision Tree,cat__ai_tool_chatgpt_gpt,0.015463,0.015463
45,Decision Tree,cat__ai_verb_assisted,0.014431,0.014431
36,Decision Tree,cat__language_python,0.012312,0.012312
49,Decision Tree,cat__ai_verb_unknown,0.010458,0.010458


## E7 — 5-fold stratified cross-validation

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
scoring = {'accuracy':'accuracy','precision':'precision','recall':'recall','f1':'f1','roc_auc':'roc_auc'}
for name, pipe in pipelines.items():
    result = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    row = {'model':name}
    for metric in scoring: row[f'{metric}_mean'] = result[f'test_{metric}'].mean(); row[f'{metric}_std'] = result[f'test_{metric}'].std()
    cv_rows.append(row)
cv_table = pd.DataFrame(cv_rows).sort_values('roc_auc_mean', ascending=False)
display(cv_table); cv_table.to_csv(REPORT_DIR / 'classification_cross_validation.csv', index=False)
print('✅ Đã lưu metrics, feature importance, confusion matrix, ROC curve và cross-validation vào reports/.')

,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
2,Random Forest,0.798095,0.002927,0.737589,0.003457,0.707943,0.007793,0.722443,0.004937,0.866192,0.005353
1,Decision Tree,0.685497,0.009413,0.570282,0.016976,0.628753,0.033313,0.597195,0.009625,0.746046,0.005560
0,Logistic Regression,0.616771,0.010043,0.483986,0.012943,0.477125,0.009437,0.480370,0.007167,0.623492,0.008664


✅ Đã lưu metrics, feature importance, confusion matrix, ROC curve và cross-validation vào reports/.
